In [18]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import pickle
from datetime import datetime

In [19]:
data = pd.read_excel("Data/out-of-sample/dataset_comp_0622.xlsx")

## 2023–mid 2025 backtesting

Extracting from yahoo finance the adjusted price for corporate splits and dividends in the period of next 3 months -> i.e. still dec 2022 composition.

In [20]:
# # Define the tickers and date range
# tickers = list(data['SYMBOL'].values) # Add more tickers as needed
# start_date = "2022-06-30"
# end_date = "2022-10-01"

# # Download the data
# yahoo_data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False, progress=False)

# # Extract adjusted close prices
# adj_close_bt = pd.DataFrame()

# # If only one ticker, yfinance doesn't use a multi-indexed DataFrame, handle accordingly
# if len(tickers) == 1:
#     adj_close_bt[tickers[0]] = yahoo_data['Adj Close']
# else:
#     for ticker in tickers:
#         try:
#             adj_close_bt[ticker] = yahoo_data[ticker]['Adj Close']
#         except KeyError:
#             print(f"No data found for {ticker}.")

# # Display the result
# print(adj_close_bt.head())

In [21]:
# #print(data.loc[data['SYMBOL'] == 'BF.B'])
# # Define the tickers and date range
# tickers = ['BF-B'] # Add more tickers as needed
# start_date = "2022-06-30"
# end_date = "2022-10-01"


# # Download the data
# data_bfb = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False, progress=False)

# # Extract adjusted close prices
# adj_close_bfb = pd.DataFrame()

# # If only one ticker, yfinance doesn't use a multi-indexed DataFrame, handle accordingly
# if len(tickers) == 1:
#     adj_close_bfb[tickers[0]] = data_bfb['BF-B']['Adj Close']
# else:
#     for ticker in tickers:
#         try:
#             adj_close_bfb[ticker] = data_bfb[ticker]['Adj Close']
#         except KeyError:
#             print(f"No data found for {ticker}.")

# # Display the result
# #print(adj_close_bfb.head())
# adj_close_bt['BF.B'] = adj_close_bfb.values

In [22]:
# price_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0622.xlsm', sheet_name='CLOSE PRICE', header = 4)
# price_next_3m.columns = price_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()


# div_rate_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0622.xlsm', sheet_name='DIV RATE', header = 4)
# div_rate_next_3m.columns = div_rate_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()


# div_date_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0622.xlsm', sheet_name='DIV DATE', header = 4)
# div_date_next_3m.columns = div_date_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()

# ffnosh_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0622.xlsm', sheet_name='FFNOSH', header = 4)
# ffnosh_next_3m.columns =ffnosh_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()



# div_date_next_3m.columns = price_next_3m.columns
# div_rate_next_3m.columns = price_next_3m.columns
# ffnosh_next_3m.columns = price_next_3m.columns

# price_next_3m.index = price_next_3m.Code
# div_rate_next_3m.index = div_rate_next_3m.Code
# div_date_next_3m.index = div_date_next_3m.Code
# ffnosh_next_3m.index = ffnosh_next_3m.Code

# price_next_3m = price_next_3m.iloc[:, 1:]
# div_rate = div_rate_next_3m.iloc[:, 1:]
# div_date_next_3m = div_date_next_3m.iloc[:, 1:]
# ffnosh_next_3m = ffnosh_next_3m.iloc[:, 1:]

# # --- Step 1: Calculate adjustment factors
# adj_factors = pd.DataFrame(1.0, index=price_next_3m.index, columns=price_next_3m.columns)

# for company in price_next_3m.columns:
#     for i in range(1, len(price_next_3m)):
#         date = price_next_3m.index[i]
#         prev_date = price_next_3m.index[i - 1]

#         # If ex-dividend happens on this day
#         if pd.notna(div_date_next_3m.at[date, company]):
#             div = div_rate.at[date, company]
#             price_prev = price_next_3m.at[prev_date, company]
#             if price_prev and price_prev != 0:
#                 factor = (price_prev - div) / price_prev
#                 adj_factors.at[date, company] = factor

# # --- Step 2: Calculate cumulative adjustment factors in reverse (like Yahoo)
# cum_factors = adj_factors.iloc[::-1].cumprod().iloc[::-1]

# # --- Step 3: Build adjusted prices
# adjusted_prices_calculated = price_next_3m * cum_factors



# for stock in ['ABMD', 'PXD', 'TWTR', 'WRK', 'DISH', 'CTXS', 'ATVI', 'SIVBQ', 'CTLT', 'NLSN', 'MRO', 'DRE','JNPR', 'DFS','SBNY']: 
#     type_stock_missing = data.loc[data['SYMBOL'] == stock, 'TYPE'].values
#     adj_close_bt[stock] = adjusted_prices_calculated.loc[adj_close_bt.index,  type_stock_missing].values

In [23]:
# adj_close_bt.to_excel("Data/out-of-sample/adj_price_yahoo_comp_0622_next_3m.xlsx")

In [24]:
adj_close_bt = pd.read_excel("Data/out-of-sample/adj_price_yahoo_comp_0622_next_3m.xlsx")
adj_close_bt.index = adj_close_bt.Date
adj_close_bt.drop(columns='Date', inplace=True)

In [25]:
nan_columns = adj_close_bt.columns[adj_close_bt.isna().any()]
print(nan_columns)

Index([], dtype='object')


In [26]:
adj_close_bt.isna().any().any()

False

### Market-cap–weighted index (S&P 500 approximation):

Start from Dec 2022 composition, but instead of freezing weights, I recalculate weights each day 

-> This way, weights “evolve naturally with returns” 

-> This mimics how the real S&P 500 works (ignoring reconstitutions)

vs 

 Buy-and-hold portfolio 

You fix the weights at Dec 2022 values (e.g., Apple 7%, Microsoft 6%).

You do not update them after that.

The portfolio return is just the weighted sum of stock returns with those fixed weights.

Over time, it drifts away from being market-cap proportional.

👉 Analogy: If you buy the S&P 500 in Dec 2022 and literally never rebalance, you’re doing buy-and-hold.

In [27]:
price_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0622.xlsm', sheet_name='CLOSE PRICE', header = 4)
price_next_3m.columns = price_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
price_next_3m = price_next_3m.iloc[:-1]

div_rate_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0622.xlsm', sheet_name='DIV RATE', header = 4)
div_rate_next_3m.columns = div_rate_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_rate_next_3m = div_rate_next_3m.iloc[:-1]

div_date_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0622.xlsm', sheet_name='DIV DATE', header = 4)
div_date_next_3m.columns = div_date_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_date_next_3m = div_date_next_3m.iloc[:-1]

ffnosh_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0622.xlsm', sheet_name='FFNOSH', header = 4)
ffnosh_next_3m.columns =ffnosh_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
ffnosh_next_3m = ffnosh_next_3m.iloc[:-1]


div_date_next_3m.columns = price_next_3m.columns
div_rate_next_3m.columns = price_next_3m.columns
ffnosh_next_3m.columns = price_next_3m.columns

price_next_3m.index = price_next_3m.Code
div_rate_next_3m.index = div_rate_next_3m.Code
div_date_next_3m.index = div_date_next_3m.Code
ffnosh_next_3m.index = ffnosh_next_3m.Code

price_next_3m = price_next_3m.iloc[:, 1:]
div_rate = div_rate_next_3m.iloc[:, 1:]
div_date_next_3m = div_date_next_3m.iloc[:, 1:]
ffnosh_next_3m = ffnosh_next_3m.iloc[:, 1:]

type_lseg = pd.read_excel("Data/out-of-sample/symbol_comp_0622.xlsm", sheet_name="SYMBOL", dtype=str).iloc[0].values[1:]
symbol_lseg = pd.read_excel("Data/out-of-sample/symbol_comp_0622.xlsm", sheet_name="SYMBOL", dtype=str, header=2)
symbol_lseg = symbol_lseg.iloc[:1].transpose().reset_index().rename(columns= {"index": "NAME", 0: "SYMBOL"}).iloc[1:]
symbol_lseg['TYPE'] = type_lseg
duplicates = symbol_lseg[symbol_lseg['SYMBOL'].duplicated()]
print(duplicates)
symbol_type_matches  = symbol_lseg.set_index("TYPE")["SYMBOL"].to_dict()

price_next_3m = price_next_3m.rename(columns=symbol_type_matches)
ffnosh_next_3m = ffnosh_next_3m.rename(columns=symbol_type_matches)


# change symbol from BRK.A to BRK-B:
price_next_3m.rename(columns={'BRK.A':'BRK-B'}, inplace=True)
ffnosh_next_3m.rename(columns={'BRK.A':'BRK-B'}, inplace=True)

price_next_3m.index = pd.to_datetime(price_next_3m.index)
ffnosh_next_3m.index = pd.to_datetime(ffnosh_next_3m.index)

# Example duplicate map
duplicate_map = {
    "GOOG": "GOOGL",  # aggregate under GOOG
    "FOX": "FOXA",    # aggregate under FOXA
    "NWS": "NWSA"     # aggregate under NWSA
}

# --- Step 1: Align indexes ---
price_next_3m.index = pd.to_datetime(price_next_3m.index)
ffnosh_next_3m.index = pd.to_datetime(ffnosh_next_3m.index)

# --- Step 2: Apply duplicate mapping ---
price_next_3m = price_next_3m.rename(columns=duplicate_map)
ffnosh_next_3m = ffnosh_next_3m.rename(columns=duplicate_map)

# After renaming, duplicates will exist → group and aggregate
def aggregate_duplicates(price_df, ffnosh_df):
    # Weighted price = Σ(price * shares) / Σ(shares)
    weighted_price = (price_df * ffnosh_df).groupby(axis=1, level=0).sum() / ffnosh_df.groupby(axis=1, level=0).sum()
    total_ffnosh = ffnosh_df.groupby(axis=1, level=0).sum()
    return weighted_price, total_ffnosh

price_agg, ffnosh_agg = aggregate_duplicates(price_next_3m, ffnosh_next_3m)



# --- Step 3: Float-adjusted market cap ---
float_mcap = price_agg * ffnosh_agg


# --- Step 4: Add sector info ---
sector_map = data.set_index("SYMBOL")["GICS Sector"].to_dict()
sector_map_series = pd.Series(sector_map)

# Columns in float_mcap but not in sector_map_series
extra_in_float_mcap = float_mcap.columns.difference(sector_map_series.index)

# Symbols in sector_map_series but not in float_mcap
extra_in_sector_map = sector_map_series.index.difference(float_mcap.columns)

print("In float_mcap but not in sector_map_series:", extra_in_float_mcap.tolist())
print("In sector_map_series but not in float_mcap:", extra_in_sector_map.tolist())

# Make sure columns align
#float_mcap = float_mcap.loc[:, sector_map_series.index]

# --- Step 5: Compute sector-level weights daily ---
sector_weights = {}
for sector in sector_map_series.unique():
    tickers = sector_map_series[sector_map_series == sector].index
    sector_mcap = float_mcap[tickers].sum(axis=1)
    sector_weights[sector] = float_mcap[tickers].div(sector_mcap, axis=0)

# Example: weights for Tech sector
print(sector_weights["Information Technology"].head())


             NAME SYMBOL    TYPE
414      NEWS 'B'   NWSA  89257J
438         FOX B   FOXA  9406MA
498  ALPHABET 'C'  GOOGL  871997
In float_mcap but not in sector_map_series: ['CEG']
In sector_map_series but not in float_mcap: []
                 IBM       AMD      ADBE       ADI      ANSS      AAPL  \
Code                                                                     
2019-01-01  0.027334  0.005655  0.031276  0.009717  0.003478  0.199685   
2019-01-02  0.027711  0.005770  0.031052  0.009729  0.003480  0.199960   
2019-01-03  0.028691  0.005520  0.031510  0.009657  0.003538  0.190206   
2019-01-04  0.028538  0.005888  0.031630  0.009469  0.003535  0.189853   
2019-01-07  0.028489  0.006319  0.031780  0.009446  0.003594  0.187777   

                AMAT      ADSK      VRSN       APH  ...      AVGO       GEN  \
Code                                                ...                       
2019-01-01  0.008780  0.008199  0.004627  0.005947  ...  0.021323  0.001463   
2019-01-02  0

/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_42718/3696991770.py:69: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  weighted_price = (price_df * ffnosh_df).groupby(axis=1, level=0).sum() / ffnosh_df.groupby(axis=1, level=0).sum()
/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_42718/3696991770.py:69: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  weighted_price = (price_df * ffnosh_df).groupby(axis=1, level=0).sum() / ffnosh_df.groupby(axis=1, level=0).sum()
/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_42718/3696991770.py:70: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  total_ffnosh = ffnosh_df.groupby(axis=1, level=0).sum()


In [28]:
for sector, w_df in sector_weights.items():
    assert all(abs(w_df.sum(axis=1) - 1) < 1e-8), f"Sector {sector} not summing to 1"

In [29]:
all_weights = pd.concat(sector_weights.values(), axis=1)
print(all_weights.sum(axis=1).head())

Code
2019-01-01    11.0
2019-01-02    11.0
2019-01-03    11.0
2019-01-04    11.0
2019-01-07    11.0
dtype: float64


In [30]:
daily_returns_next_3m = adj_close_bt.pct_change().dropna()

In [31]:
# Columns in float_mcap but not in sector_map_series
extra_in_float_mcap = float_mcap.columns.difference(daily_returns_next_3m.columns)


print("In float_mcap but not in daily_returns:", extra_in_float_mcap.tolist())



In float_mcap but not in daily_returns: ['CEG']


In [32]:
sector_portfolio_returns = {}

for sector, weights_df in sector_weights.items():
    # Align weights with returns (common tickers + dates)
    common_tickers = weights_df.columns.intersection(daily_returns_next_3m.columns)
    common_index = weights_df.index.intersection(daily_returns_next_3m.index)

    w = weights_df.loc[common_index, common_tickers]
    r = daily_returns_next_3m.loc[common_index, common_tickers]

    # Multiply elementwise and sum across tickers → portfolio return per day
    sector_returns = (w * r).sum(axis=1)

    sector_portfolio_returns[sector] = sector_returns

# Put all in a single DataFrame
sector_portfolio_df = pd.DataFrame(sector_portfolio_returns)
print(sector_portfolio_df.head())


            Consumer Discretionary  Health Care  Utilities  \
2022-07-01                0.020503     0.012264   0.025668   
2022-07-05                0.022952    -0.005812  -0.033608   
2022-07-06               -0.002397     0.006706   0.010007   
2022-07-07                0.025557     0.004706  -0.001523   
2022-07-08                0.000541     0.002818  -0.003514   

            Information Technology  Real Estate  Materials  Industrials  \
2022-07-01                0.001545     0.018545   0.007120     0.010165   
2022-07-05                0.013331    -0.004879  -0.019485    -0.013802   
2022-07-06                0.009601     0.000238   0.006212     0.007211   
2022-07-07                0.022400     0.000640   0.009306     0.011284   
2022-07-08                0.001027    -0.005475  -0.009819    -0.004225   

            Financials    Energy  Communication Services  Consumer Staples  
2022-07-01    0.014249  0.014334                0.006496          0.013928  
2022-07-05   -0.000681

In [33]:
def compute_fixed_weight_returns(returns_df, weights, stock_labels):
    """
    Compute fixed-weight portfolio returns (decarbonised portfolio).
    
    Args:
        returns_df: DataFrame of monthly returns
        weights: np.array of optimised weights
        stock_labels: list or Index of stock tickers

    Returns:
        Series of portfolio returns over time
    """
    sector_returns = returns_df[stock_labels]
    portfolio_returns = sector_returns.dot(weights)
    return portfolio_returns

In [34]:
with open('Data/out-of-sample/optimal_portfolios_0622_minimum_weight.pkl', 'rb') as f:
    optimal_portfolios_shrink = pickle.load(f)

In [35]:
decarb_sector_returns = {}

for sector, data in optimal_portfolios_shrink.items():
    w_opt = data['w_opt']
    tickers = data['stock_labels']
    
    sector_returns = compute_fixed_weight_returns(
        daily_returns_next_3m,
        weights=w_opt,
        stock_labels=tickers
    )
    
    decarb_sector_returns[sector] = sector_returns

In [36]:
benchmark_sectors_daily_returns = sector_portfolio_df
optimised_sectors_daily_returns = pd.DataFrame(decarb_sector_returns)

In [37]:
import numpy as np
import pandas as pd

sector_te = {}

for sector in benchmark_sectors_daily_returns.columns:
    # Align indices
    r_b = benchmark_sectors_daily_returns[sector]
    r_d = optimised_sectors_daily_returns[sector]
    common_idx = r_b.index.intersection(r_d.index)
    r_b, r_d = r_b.loc[common_idx], r_d.loc[common_idx]

    # Active returns
    active = r_d - r_b

    # Daily TE → annualised
    te_daily = active.std()
    te_ann = te_daily * np.sqrt(252)

    sector_te[sector] = te_ann

# Build DataFrame
te_df = pd.DataFrame.from_dict(sector_te, orient="index", columns=["Annualised_TE"])
print(te_df.sort_values("Annualised_TE", ascending=False).round(4))


                        Annualised_TE
Financials                     0.0289
Utilities                      0.0263
Health Care                    0.0253
Industrials                    0.0239
Materials                      0.0239
Consumer Discretionary         0.0230
Real Estate                    0.0230
Information Technology         0.0227
Consumer Staples               0.0212
Energy                         0.0188
Communication Services         0.0165


In [38]:
# Start with 1 and compound returns
benchmark_cum = (1 + benchmark_sectors_daily_returns).cumprod()
optimised_cum = (1 + optimised_sectors_daily_returns).cumprod()

initial_investment = 100_000

benchmark_final_values = benchmark_cum.iloc[-1] * initial_investment
optimised_final_values = optimised_cum.iloc[-1] * initial_investment

difference = optimised_final_values - benchmark_final_values

final_df = pd.DataFrame({
    "Benchmark_Final": benchmark_final_values,
    "Optimised_Final": optimised_final_values,
    "Difference": difference
}).round(2)

print(final_df.sort_values("Difference", ascending=False))


                        Benchmark_Final  Optimised_Final  Difference
Real Estate                    88359.58         92174.92     3815.34
Health Care                    97251.85         96482.75     -769.10
Energy                        104252.10        102921.68    -1330.43
Materials                      94803.57         93112.66    -1690.91
Communication Services         89743.60         87827.52    -1916.08
Information Technology         96501.01         94580.67    -1920.34
Financials                     98259.97         96243.52    -2016.45
Utilities                      95230.43         92839.40    -2391.03
Consumer Staples               96080.72         93406.66    -2674.06
Industrials                    98227.37         95530.36    -2697.01
Consumer Discretionary        108431.07        105022.73    -3408.34


In [39]:
import pandas as pd

# --- Daily sector weights ---
sector_sp500_weights = float_mcap.groupby(sector_map_series, axis=1).sum()
sector_sp500_weights = sector_sp500_weights.div(sector_sp500_weights.sum(axis=1), axis=0)

# --- Cumulative growth per sector ---
benchmark_cum = (1 + benchmark_sectors_daily_returns).cumprod()
optimised_cum = (1 + optimised_sectors_daily_returns).cumprod()

# --- Start with $100k ---
initial_investment = 100_000

# Align
benchmark_cum, sector_sp500_weights = benchmark_cum.align(sector_sp500_weights, join="inner", axis=0)
optimised_cum, sector_sp500_weights = optimised_cum.align(sector_sp500_weights, join="inner", axis=0)

# --- Portfolio values (daily evolving) ---
benchmark_portfolio_value = (benchmark_cum * sector_sp500_weights).sum(axis=1)
optimised_portfolio_value = (optimised_cum * sector_sp500_weights).sum(axis=1)

# Scale both portfolios to same baseline (benchmark scaling)
scale_factor = initial_investment / benchmark_portfolio_value.iloc[0]
benchmark_portfolio_value *= scale_factor
optimised_portfolio_value *= scale_factor

# --- Final results ---
final_benchmark = benchmark_portfolio_value.iloc[-1]
final_optimised = optimised_portfolio_value.iloc[-1]
difference = final_optimised - final_benchmark


# --- Sector-level portfolio values ---
benchmark_sector_values = benchmark_cum * sector_sp500_weights * scale_factor
optimised_sector_values = optimised_cum * sector_sp500_weights * scale_factor

# --- Final sector contributions ---
final_benchmark_by_sector = benchmark_sector_values.iloc[-1]
final_optimised_by_sector = optimised_sector_values.iloc[-1]

sector_contributions = final_optimised_by_sector - final_benchmark_by_sector

# Contributions in $
sector_contributions_dollars = sector_contributions

# Contributions in % of total difference
sector_contributions_pct = sector_contributions_dollars / difference * 100

# Build results table
sector_results = pd.DataFrame({
    "Contribution ($)": sector_contributions_dollars,
    "Contribution (%)": sector_contributions_pct
})

# Add totals
sector_results.loc["Total"] = [difference, 100.0]                      # sector shares sum to 100%

# Format nicely
print(sector_results.round(2).sort_values("Contribution ($)", ascending=False))


portfolio_summary = pd.DataFrame({
    "Final value ($)": [final_benchmark, final_optimised],
    "Return (%)": [
        (final_benchmark / initial_investment - 1) * 100,
        (final_optimised / initial_investment - 1) * 100
    ]
}, index=["Benchmark", "Decarbonised"])

# Add row for difference
portfolio_summary.loc["Difference"] = [
    difference,
    (final_optimised / final_benchmark - 1) * 100
]

print(portfolio_summary.round(2))


                        Contribution ($)  Contribution (%)
Real Estate                        75.44            -14.94
Health Care                        64.41            -12.75
Information Technology            -10.81              2.14
Utilities                         -15.85              3.14
Energy                            -18.97              3.76
Materials                         -33.03              6.54
Communication Services            -45.15              8.94
Consumer Staples                  -76.26             15.10
Financials                       -112.31             22.24
Industrials                      -135.04             26.73
Consumer Discretionary           -197.52             39.11
Total                            -505.09            100.00
              Final value ($)  Return (%)
Benchmark            96774.74       -3.23
Decarbonised         96269.64       -3.73
Difference            -505.09       -0.52


/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_42718/3450537844.py:4: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  sector_sp500_weights = float_mcap.groupby(sector_map_series, axis=1).sum()


## Rerunning TE estimation and comparing benchmark vs decarbonised portfolios, now including stocks (OGN, CARR, OTIS) that were excluded earlier in the benchmark because of missing return data in the covariance period

In [40]:
price_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0622.xlsm', sheet_name='CLOSE PRICE', header = 4)
price_next_3m.columns = price_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
price_next_3m = price_next_3m.iloc[:-1]

div_rate_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0622.xlsm', sheet_name='DIV RATE', header = 4)
div_rate_next_3m.columns = div_rate_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_rate_next_3m = div_rate_next_3m.iloc[:-1]

div_date_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0622.xlsm', sheet_name='DIV DATE', header = 4)
div_date_next_3m.columns = div_date_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_date_next_3m = div_date_next_3m.iloc[:-1]

ffnosh_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0622.xlsm', sheet_name='FFNOSH', header = 4)
ffnosh_next_3m.columns =ffnosh_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
ffnosh_next_3m = ffnosh_next_3m.iloc[:-1]

div_date_next_3m.columns = price_next_3m.columns
div_rate_next_3m.columns = price_next_3m.columns
ffnosh_next_3m.columns = price_next_3m.columns

price_next_3m.index = price_next_3m.Code
div_rate_next_3m.index = div_rate_next_3m.Code
div_date_next_3m.index = div_date_next_3m.Code
ffnosh_next_3m.index = ffnosh_next_3m.Code

price_next_3m = price_next_3m.iloc[:, 1:]
div_rate = div_rate_next_3m.iloc[:, 1:]
div_date_next_3m = div_date_next_3m.iloc[:, 1:]
ffnosh_next_3m = ffnosh_next_3m.iloc[:, 1:]

type_lseg = pd.read_excel("Data/out-of-sample/symbol_comp_0622.xlsm", sheet_name="SYMBOL", dtype=str).iloc[0].values[1:]
symbol_lseg = pd.read_excel("Data/out-of-sample/symbol_comp_0622.xlsm", sheet_name="SYMBOL", dtype=str, header=2)
symbol_lseg = symbol_lseg.iloc[:1].transpose().reset_index().rename(columns= {"index": "NAME", 0: "SYMBOL"}).iloc[1:]
symbol_lseg['TYPE'] = type_lseg

symbol_type_matches  = symbol_lseg.set_index("TYPE")["SYMBOL"].to_dict()

price_next_3m = price_next_3m.rename(columns=symbol_type_matches)
ffnosh_next_3m = ffnosh_next_3m.rename(columns=symbol_type_matches)


# change symbol from BRK.A to BRK-B:
price_next_3m.rename(columns={'BRK.A':'BRK-B'}, inplace=True)
ffnosh_next_3m.rename(columns={'BRK.A':'BRK-B'}, inplace=True)

price_next_3m.index = pd.to_datetime(price_next_3m.index)
ffnosh_next_3m.index = pd.to_datetime(ffnosh_next_3m.index)

# Example duplicate map
duplicate_map = {
    "GOOG": "GOOGL",  # aggregate under GOOG
    "FOX": "FOXA",    # aggregate under FOXA
    "NWS": "NWSA"     # aggregate under NWSA
}

# --- Step 1: Align indexes ---
price_next_3m.index = pd.to_datetime(price_next_3m.index)
ffnosh_next_3m.index = pd.to_datetime(ffnosh_next_3m.index)

# --- Step 2: Apply duplicate mapping ---
price_next_3m = price_next_3m.rename(columns=duplicate_map)
ffnosh_next_3m = ffnosh_next_3m.rename(columns=duplicate_map)

# After renaming, duplicates will exist → group and aggregate
def aggregate_duplicates(price_df, ffnosh_df):
    # Weighted price = Σ(price * shares) / Σ(shares)
    weighted_price = (price_df * ffnosh_df).groupby(axis=1, level=0).sum() / ffnosh_df.groupby(axis=1, level=0).sum()
    total_ffnosh = ffnosh_df.groupby(axis=1, level=0).sum()
    return weighted_price, total_ffnosh


price_agg, ffnosh_agg = aggregate_duplicates(price_next_3m, ffnosh_next_3m)

# --- Step 3: Float-adjusted market cap ---
float_mcap = price_agg * ffnosh_agg

data = pd.read_excel("Data/out-of-sample/dataset_comp_0622.xlsx")
# --- Step 4: Add sector info ---
sector_map = data.set_index("SYMBOL")["GICS Sector"].to_dict()

sector_map.update({'CEG': 'Utilities'})

sector_map_series = pd.Series(sector_map)

# Columns in float_mcap but not in sector_map_series
extra_in_float_mcap = float_mcap.columns.difference(sector_map_series.index)

# Symbols in sector_map_series but not in float_mcap
extra_in_sector_map = sector_map_series.index.difference(float_mcap.columns)

print("In float_mcap but not in sector_map_series:", extra_in_float_mcap.tolist())
print("In sector_map_series but not in float_mcap:", extra_in_sector_map.tolist())

# Make sure columns align
#float_mcap = float_mcap.loc[:, sector_map_series.index]

# --- Step 5: Compute sector-level weights daily ---
sector_weights = {}
for sector in sector_map_series.unique():
    tickers = sector_map_series[sector_map_series == sector].index
    sector_mcap = float_mcap[tickers].sum(axis=1)
    sector_weights[sector] = float_mcap[tickers].div(sector_mcap, axis=0)

sector_portfolio_returns = {}

for sector, weights_df in sector_weights.items():
    # Align weights with returns (common tickers + dates)
    common_tickers = weights_df.columns.intersection(daily_returns_next_3m.columns)
    common_index = weights_df.index.intersection(daily_returns_next_3m.index)

    w = weights_df.loc[common_index, common_tickers]
    r = daily_returns_next_3m.loc[common_index, common_tickers]

    # Multiply elementwise and sum across tickers → portfolio return per day
    sector_returns = (w * r).sum(axis=1)

    sector_portfolio_returns[sector] = sector_returns

# Put all in a single DataFrame
sector_portfolio_df = pd.DataFrame(sector_portfolio_returns)


# --- Daily sector weights ---
sector_sp500_weights = float_mcap.groupby(sector_map_series, axis=1).sum()
sector_sp500_weights = sector_sp500_weights.div(sector_sp500_weights.sum(axis=1), axis=0)

# --- Cumulative growth per sector ---
benchmark_cum = (1 + benchmark_sectors_daily_returns).cumprod()
optimised_cum = (1 + optimised_sectors_daily_returns).cumprod()

# --- Start with $100k ---
initial_investment = 100_000

# Align
benchmark_cum, sector_sp500_weights = benchmark_cum.align(sector_sp500_weights, join="inner", axis=0)
optimised_cum, sector_sp500_weights = optimised_cum.align(sector_sp500_weights, join="inner", axis=0)

# --- Portfolio values (daily evolving) ---
benchmark_portfolio_value = (benchmark_cum * sector_sp500_weights).sum(axis=1)
optimised_portfolio_value = (optimised_cum * sector_sp500_weights).sum(axis=1)

# Scale both portfolios to same baseline (benchmark scaling)
scale_factor = initial_investment / benchmark_portfolio_value.iloc[0]
benchmark_portfolio_value *= scale_factor
optimised_portfolio_value *= scale_factor

# --- Final results ---
final_benchmark = benchmark_portfolio_value.iloc[-1]
final_optimised = optimised_portfolio_value.iloc[-1]
difference = final_optimised - final_benchmark


# --- Sector-level portfolio values ---
benchmark_sector_values = benchmark_cum * sector_sp500_weights * scale_factor
optimised_sector_values = optimised_cum * sector_sp500_weights * scale_factor

# --- Final sector contributions ---
final_benchmark_by_sector = benchmark_sector_values.iloc[-1]
final_optimised_by_sector = optimised_sector_values.iloc[-1]

sector_contributions = final_optimised_by_sector - final_benchmark_by_sector

# Contributions in $
sector_contributions_dollars = sector_contributions

# Contributions in % of total difference
sector_contributions_pct = sector_contributions_dollars / difference * 100

# Build results table
sector_results = pd.DataFrame({
    "Contribution ($)": sector_contributions_dollars,
    "Contribution (%)": sector_contributions_pct
})

# Add totals
sector_results.loc["Total"] = [difference, 100.0]                      # sector shares sum to 100%

# Format nicely
print(sector_results.round(2).sort_values("Contribution ($)", ascending=False))


portfolio_summary = pd.DataFrame({
    "Final value ($)": [final_benchmark, final_optimised],
    "Return (%)": [
        (final_benchmark / initial_investment - 1) * 100,
        (final_optimised / initial_investment - 1) * 100
    ]
}, index=["Benchmark", "Decarbonised"])

# Add row for difference
portfolio_summary.loc["Difference"] = [
    difference,
    (final_optimised / final_benchmark - 1) * 100
]

print(portfolio_summary.round(2))
sector_results.to_excel("Data/te-testing-results/benchmark_vs_decarb_sectoral_contribution_proportional_0622.xlsx")
portfolio_summary.to_excel("Data/te-testing-results/benchmark_vs_decarb_final_value_proportional_0622.xlsx")

/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_42718/3548512302.py:67: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  weighted_price = (price_df * ffnosh_df).groupby(axis=1, level=0).sum() / ffnosh_df.groupby(axis=1, level=0).sum()
/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_42718/3548512302.py:67: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  weighted_price = (price_df * ffnosh_df).groupby(axis=1, level=0).sum() / ffnosh_df.groupby(axis=1, level=0).sum()
/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_42718/3548512302.py:68: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  total_ffnosh = ffnosh_df.groupby(axis=1, level=0).sum()
/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_42718/3548512302.py:124: FutureWarning: DataFrame.groupby with axis=1 is depre

In float_mcap but not in sector_map_series: []
In sector_map_series but not in float_mcap: []
                        Contribution ($)  Contribution (%)
Real Estate                        75.38            -14.92
Health Care                        64.36            -12.74
Information Technology            -10.80              2.14
Utilities                         -16.26              3.22
Energy                            -18.95              3.75
Materials                         -33.01              6.54
Communication Services            -45.12              8.93
Consumer Staples                  -76.20             15.09
Financials                       -112.22             22.22
Industrials                      -134.93             26.71
Consumer Discretionary           -197.37             39.07
Total                            -505.11            100.00
              Final value ($)  Return (%)
Benchmark            96772.02       -3.23
Decarbonised         96266.91       -3.73
Difference   

In [41]:
benchmark_sectors_daily_returns = sector_portfolio_df
optimised_sectors_daily_returns = pd.DataFrame(decarb_sector_returns)

In [42]:
import numpy as np
import pandas as pd

sector_te = {}

for sector in benchmark_sectors_daily_returns.columns:
    # Align indices
    r_b = benchmark_sectors_daily_returns[sector]
    r_d = optimised_sectors_daily_returns[sector]
    common_idx = r_b.index.intersection(r_d.index)
    r_b, r_d = r_b.loc[common_idx], r_d.loc[common_idx]

    # Active returns
    active = r_d - r_b

    # Daily TE → annualised
    te_daily = active.std()
    te_ann = te_daily * np.sqrt(252)

    sector_te[sector] = te_ann

# Build DataFrame
te_df = pd.DataFrame.from_dict(sector_te, orient="index", columns=["Annualised_TE"])
print(te_df.sort_values("Annualised_TE", ascending=False).round(4))
te_df.to_excel("Data/te-testing-results/te_0622.xlsx")

                        Annualised_TE
Financials                     0.0289
Utilities                      0.0271
Health Care                    0.0253
Industrials                    0.0239
Materials                      0.0239
Consumer Discretionary         0.0230
Real Estate                    0.0230
Information Technology         0.0227
Consumer Staples               0.0212
Energy                         0.0188
Communication Services         0.0165


In [43]:
# Start with 1 and compound returns
benchmark_cum = (1 + benchmark_sectors_daily_returns).cumprod()
optimised_cum = (1 + optimised_sectors_daily_returns).cumprod()

initial_investment = 100_000

# Final values by sector
benchmark_final_values = benchmark_cum.iloc[-1] * initial_investment
optimised_final_values = optimised_cum.iloc[-1] * initial_investment

# Differences in final $ values
difference = optimised_final_values - benchmark_final_values

# Returns relative to initial investment
benchmark_returns = (benchmark_final_values / initial_investment - 1) * 100
optimised_returns = (optimised_final_values / initial_investment - 1) * 100
difference_returns = optimised_returns - benchmark_returns

# Build DataFrame
final_df = pd.DataFrame({
    "Benchmark_Final": benchmark_final_values,
    "Optimised_Final": optimised_final_values,
    "Difference ($)": difference,
    "Benchmark_Return (%)": benchmark_returns,
    "Optimised_Return (%)": optimised_returns,
    "Difference_Return (%)": difference_returns
}).round(2)

print(final_df.sort_values("Difference ($)", ascending=False))
final_df.to_excel("Data/te-testing-results/benchmark_vs_decarb_final_value_equal_weight_0622.xlsx")

                        Benchmark_Final  Optimised_Final  Difference ($)  \
Real Estate                    88359.58         92174.92         3815.34   
Health Care                    97251.85         96482.75         -769.10   
Energy                        104252.10        102921.68        -1330.43   
Materials                      94803.57         93112.66        -1690.91   
Communication Services         89743.60         87827.52        -1916.08   
Information Technology         96501.01         94580.67        -1920.34   
Financials                     98259.97         96243.52        -2016.45   
Utilities                      95403.74         92839.40        -2564.34   
Consumer Staples               96080.72         93406.66        -2674.06   
Industrials                    98227.37         95530.36        -2697.01   
Consumer Discretionary        108431.07        105022.73        -3408.34   

                        Benchmark_Return (%)  Optimised_Return (%)  \
Real Estate      